# Synthèse multidimensionnelle — le catalogue light / standard / full

Ce carnet est le dernier de la sous-série *exécution* : après les modes
d'orchestration (`07c`) et la fiche d'identité d'un argument (`08e`), il porte
le **fil conducteur du catalogue de niveaux** — quelles dimensions d'analyse
activer à quel niveau — et la **synthèse en 3 actes** qui en restitue le
résultat.

Le catalogue vient de l'organe `workflow_levels.py` (étape 5 du volet B,
EPIC #4960) : un port **déterministe** de la recette pédagogique du catalogue
EPITA `orchestration/workflows.py`. Tout ce que ce carnet exécute est
rejouable hors ligne, à l'identique — aucun appel LLM, aucune clé API.

## Objectifs

1. **Lire le catalogue** : les 9 dimensions, leur niveau minimal, l'organe
   que chacune invoque (provenance jamais cachée).
2. **Exécuter les trois niveaux** sur le banc embarqué et observer la
   gradation du fill-rate — la part de l'état d'analyse que chaque niveau
   peuple.
3. **Croiser avec le sas EPITA** : les mesures du dépôt étudiant
   (`orchestration_modes.py`) donnaient des DAG de 3/15/17 phases LLM ;
   ici 3/6/9 dimensions déterministes. Les deux échelles ne sont pas
   communes — et c'est documenté, pas maquillé.

## Deux mondes, un même fil conducteur

| | Sas EPITA (`07c`, mesures embarquées) | Ce carnet (organe `workflow_levels`) |
|---|---|---|
| Moteur | DAG d'agents LLM (+ Tweety) | organes déterministes de la série |
| Niveaux | light 3 / standard 15 / full 17 **phases** | light 3 / standard 6 / full 9 **dimensions** |
| Coût par run | 0,55 / 0,95 USD (BYOK mesuré) | 0 — rejouable hors ligne |
| Fill de l'état | 0,179 / 0,462 / 0,487 (LLM) | mesuré ci-dessous (déterministe) |

L'emboîtement est strict dans les deux cas : light ⊂ standard ⊂ full. Les
constantes 3/15/17 sont portées par `orchestration_modes.DEPTH_PARITY` ; les
compteurs 3/6/9 sont **dérivés** du catalogue lui-même (aucun littéral durci).

In [1]:
import sys
from pathlib import Path

# La serie vit dans ce dossier ; argumentation_lib et workflow_levels y resident.
BASE = Path.cwd()
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

from argumentation_lib import initialize_jvm, is_jvm_started

# Fail-loud : la serie refuse le mode degrade (anti-theatre, 00-Setup).
ok = initialize_jvm(verbose=False)
print("JVM demarree :", ok, "| is_jvm_started :", is_jvm_started())
assert ok, "JVM non demarree : verifier Tweety/libs (jar 7a-java8-shade)"

import spacy
print("spaCy :", spacy.__version__)

JVM demarree : True | is_jvm_started : True


spaCy : 3.8.16


In [2]:
import workflow_levels as wl
import orchestration_modes as om

print("Niveaux :", wl.LEVELS)
print("Dimensions au catalogue :", len(wl.DIMENSIONS))
print("Parity note :", wl.PARITY_NOTE.format(
    n="3/6/9", detail="emboitement strict"
)[:120], "...")

Niveaux : ('light', 'standard', 'full')
Dimensions au catalogue : 9
Parity note : Catalogue deterministe : 3/6/9 dimensions par niveau (emboitement strict). DAG LLM de la source : 3/15/17 phases (consta ...


## Le banc : un débat municipal, annoté

Le banc embarque un court débat sur la piétonisation d'un centre-ville, et
**ses annotations formelles déclarées** : traduction PL, AF de Dung, formule
de décision. Le passage du texte naturel à ces structures n'est **pas**
automatique ici — le gap NL→structure est documenté comme non bouché
déterministement (body #4960) ; les formules sont des artefacts du banc,
déclarés comme tels.

In [3]:
print(wl.BENCH["raw_text"])

Le conseil municipal doit interdire les echoppes du centre-ville, car elles detruisent le commerce de proximite. Si les echoppes detruisent le commerce, alors les boutiques centrales fermeront. On ne peut pas faire confiance à ce conseiller qui defend les echoppes. Selon une etude de boutique, la mesure est efficace. On a toujours fait comme ça, pourquoi changer la position de la ville ? Le premier pas vers le declin du centre-ville. Toute regulation qui protege le commerce de proximite merite d'etre essayee, et la fermeture des echoppes protege le commerce de proximite. Donc la fermeture des echoppes merite d'etre essayee.


In [4]:
print("KB PL (3 formules) :")
print(wl.BENCH["pl_belief_set"])
print()
print("AF Dung — arguments :", wl.BENCH["dung_arguments"])
print("AF Dung — attaques  :", wl.BENCH["dung_attacks"])
print()
print("Formule de decision :", wl.BENCH["decision_formula"])

KB PL (3 formules) :
detruit_proximite
detruit_proximite => fermeture_boutiques
fermeture_echoppes => protege_proximite

AF Dung — arguments : ['A1_interdire', 'A2_evidence', 'A3_fermeture', 'A4_ad_hominem', 'A5_regulation']
AF Dung — attaques  : [['A4_ad_hominem', 'A1_interdire'], ['A2_evidence', 'A4_ad_hominem'], ['A5_regulation', 'A3_fermeture']]

Formule de decision : fermeture_boutiques


### Lecture du résultat : ce que le banc embarque

Le débat porte quatre sophismes — et pas n'importe lesquels : ses phrases sont
les **formes canoniques** mesurées sur l'organe de détection symbolique
(`fallacy_rules`, #17348). Sur le corpus de référence de cet organe, 4 formes
sur 9 matchent ; le banc n'embarque que des formes qui matchent, pour que la
dimension sophismes du niveau standard ait de la matière réelle. L'AF, elle,
condense le débat en 5 arguments et 3 attaques : l'attaque *ad hominem* y
est un argument qui en attaque un autre — exactement comme Dung le formalise.

In [5]:
print(f"{'dimension':<22} {'niveau min':<12} organes")
print("-" * 88)
for d in wl.DIMENSIONS:
    print(f"{d.name:<22} {d.level_min:<12} {d.organ}")

dimension              niveau min   organes
----------------------------------------------------------------------------------------
extract_arguments      light        releve deterministe (marqueurs)
pl_belief_set          light        Tweety PlParser + SimplePlReasoner (pattern 00-Setup)
pl_queries             light        Tweety SimplePlReasoner (requetes)
informal_fallacies     standard     fallacy_rules.py — organe symbolique #17348
dung_semantics         standard     Tweety Simple{Grounded,Preferred,Stable}Reasoner (Tweety-05)
fol_consistency        standard     DECLARE INDISPONIBLE (divergence 5 : jar sans logics.fol)
modal_queries          full         DECLAREE INDISPONIBLE (divergence 5 : jar sans logics.ml)
sat_decision           full         Tweety SimplePlReasoner sur la formule de decision
synthesis_3_actes      full         format du contrat 3 actes (carnet 08d)


### Lecture du résultat : le catalogue et ses organes

Chaque dimension cite l'organe qu'elle invoque — les classes Tweety réelles
(`Simple*Reasoner`, noms mesurés dans le jar), l'organe symbolique de la
série (`fallacy_rules`), ou un état **déclaré** : deux dimensions du
catalogue (FOL, modale) sont annoncées **indisponibles dans cet arbre**, avec
leur preuve. C'est la différence entre un catalogue qui prétend et un
catalogue qui prouve.

In [6]:
comptages = [len(wl.dimensions_at(niveau)) for niveau in wl.LEVELS]
print("Dimensions actives :", dict(zip(wl.LEVELS, comptages)))
assert comptages == [3, 6, 9], "l'emboitement est structurel, pas un litteral"

Dimensions actives : {'light': 3, 'standard': 6, 'full': 9}


## Niveau light : relever et requêter

Trois dimensions : le relevé des arguments (marqueurs déterministes), la KB
PL posée, la requête Tweety. C'est le niveau « lecteur pressé » : il répond à
*que dit ce texte, et que peut-on en dériver formellement ?*

In [7]:
etat_light, run_light = wl.run_level(wl.BENCH, "light")
print(run_light.verdict_band())
for note in run_light.notes:
    print(" -", note)
print("Reponses aux requetes :", etat_light.answers)

[light] 3 dimensions executees, fill 0.71 (5/7 champs peuples)


 - extract_arguments: 4 segments releves (marqueurs deterministes)
 - pl_belief_set: KB PL coherente ; 1 requete(s) Tweety (fermeture_boutiques)
 - pl_queries: KB PL coherente ; 1 requete(s) Tweety (fermeture_boutiques)
Reponses aux requetes : {'fermeture_boutiques': 'True'}


### Lecture du résultat : light

Le fill-rate monte à 0,71 : cinq champs de l'état sur sept sont peuplés. La
requête `fermeture_boutiques` est **dérivée** (réponse vraie) dès ce niveau —
la KB embarque la chaîne complète (fait + règles), et `SimplePlReasoner`
n'a besoin de rien d'autre. Ce que le light ne fait pas encore : aucun
sophisme relevé, aucune sémantique d'attaque — les champs restent vides,
et c'est exactement ce que le fill-report montre.

### Exercice 1 — Prédire le relevé

La dimension `extract_arguments` retient les phrases portant un marqueur
d'argumentation (`car`, `donc`, `alors`, `si`, `toute`, `tout le monde`,
`merite`). Ajoutez au banc une phrase **sans** marqueur puis une phrase
**avec**, et prédisez le compte de segments avant de relancer le niveau
light.

In [8]:
# Exercice a completer
# 1. Construire banc_ex1 = dict(wl.BENCH) avec raw_text modifie :
#    - une phrase sans marqueur (ex. "Les echoppes durent toute l'annee.")
#    - une phrase avec marqueur (ex. "... car la ville doit vivre.")
# 2. Predire le nombre de segments, ecrire la prediction ici.
# 3. Relancer wl.run_level(banc_ex1, "light") et comparer.
prediction = None  # TODO etudiant : nombre de segments attendu
# ... votre code ...
print("Exercice a completer")

Exercice a completer


## Niveau standard : attaquer

Le standard ajoute la détection de sophismes (organe symbolique) et
l'analyse d'acceptabilité de Dung (trois sémantiques, classes Tweety
réelles). Le texte cesse d'être une suite d'affirmations : il devient un
graphe d'attaques dont on calcule ce qui **tient**.

In [9]:
etat_std, run_std = wl.run_level(wl.BENCH, "standard")
print(run_std.verdict_band())
for note in run_std.notes:
    print(" -", note)

[standard] 5 dimensions executees, 1 declarees indisponibles, fill 0.86 (6/7 champs peuples)
 - extract_arguments: 4 segments releves (marqueurs deterministes)
 - pl_belief_set: KB PL coherente ; 1 requete(s) Tweety (fermeture_boutiques)
 - pl_queries: KB PL coherente ; 1 requete(s) Tweety (fermeture_boutiques)
 - informal_fallacies: 4 sophisme(s) releve(s) par motifs symboliques (AD_HOMINEM_DIRECT, APPEL_A_LA_TRADITION, ARGUMENT_AUTORITE, PENTE_GLISSANTE)
 - dung_semantics: AF sous 3 semantiques : {'grounded': 1, 'preferred': 1, 'stable': 1}
 - fol_consistency: INDISPONIBLE — organe FOL absent : 0 classe logics.fol dans le jar 7a-java8-shade (inventaire zipfile) ; handler porte en dependances non vendues (prover9, TweetyInitializer C186f) — note G.9 de argumentation_lib


### Lecture du résultat : standard

Quatre sophismes relevés, sur quatre clés distinctes — les formes canoniques
du banc ont toutes matché. L'AF sous trois sémantiques rend : une extension
**grounded** (la plus prudente — ce qui survit à tout doute), et les
extensions preferred/stable. La note `1 declaree(s) indisponible(s)` est la
dimension FOL : le catalogue l'annonce, le run la rend avec sa preuve — on
l'ouvre au niveau full.

In [10]:
for f in etat_std.identified_fallacies:
    print(f"{f['key']:<24} {f['excerpt'][:52]}")

AD_HOMINEM_DIRECT        On ne peut pas faire confiance à ce conseiller
ARGUMENT_AUTORITE        Selon une etude de
APPEL_A_LA_TRADITION     On a toujours fait comme ça
PENTE_GLISSANTE          Le premier pas vers le


### Lecture du résultat : les quatre sophismes

Chaque détection porte sa clé, son extrait et sa provenance
(`fallacy_rules#17348`). La limite de l'organe est mesurée et assumée : ses
motifs spaCy attrapent les formes canoniques, pas les paraphrases — sur le
corpus de référence, 4 formes sur 9 matchent. La taxonomie complète (1552
sophismes, pilotée LLM dans le sas EPITA) n'est pas standalone-importable
dans cet arbre : sa dépendance SK amont n'est pas vendue. C'est la
divergence 6 du contrat de fidélité de l'organe.

### Exercice 2 — Faire tomber un argument

L'AF du banc porte 3 attaques. Ajoutez une attaque de `A3_fermeture` vers
`A1_interdire` et relancez le niveau standard : comment l'extension grounded
change-t-elle ? Un argument attaqué non défendu n'appartient jamais à
l'extension grounded — prédisez lesquels survivent avant de exécuter.

In [11]:
# Exercice a completer
# 1. banc_ex2 = dict(wl.BENCH) avec dung_attacks + ["A3_fermeture", "A1_interdire"]
# 2. Predire le contenu de l'extension grounded.
grounded_attendu = None  # TODO etudiant : liste de noms d'arguments attendus
# 3. Relancer wl.run_level(banc_ex2, "standard") et lire
#    etat.belief_sets["dung_extensions"].
print("Exercice a completer")

Exercice a completer


## Niveau full : décider, synthétiser — et prouver les absences

Le full ajoute la décision (la formule d'engagement interrogée sur la KB
complète) et la synthèse 3 actes. Il porte aussi les deux dimensions
**déclarées indisponibles** — FOL et modale — dont l'absence d'organe est
prouvée ci-dessous par l'inventaire du jar.

In [12]:
etat_full, run_full = wl.run_level(wl.BENCH, "full")
print(run_full.verdict_band())
for note in run_full.notes:
    print(" -", note)

[full] 7 dimensions executees, 2 declarees indisponibles, fill 1.00 (7/7 champs peuples)
 - extract_arguments: 4 segments releves (marqueurs deterministes)
 - pl_belief_set: KB PL coherente ; 1 requete(s) Tweety (fermeture_boutiques)
 - pl_queries: KB PL coherente ; 1 requete(s) Tweety (fermeture_boutiques)
 - informal_fallacies: 4 sophisme(s) releve(s) par motifs symboliques (AD_HOMINEM_DIRECT, APPEL_A_LA_TRADITION, ARGUMENT_AUTORITE, PENTE_GLISSANTE)
 - dung_semantics: AF sous 3 semantiques : {'grounded': 1, 'preferred': 1, 'stable': 1}
 - fol_consistency: INDISPONIBLE — organe FOL absent : 0 classe logics.fol dans le jar 7a-java8-shade (inventaire zipfile) ; handler porte en dependances non vendues (prover9, TweetyInitializer C186f) — note G.9 de argumentation_lib
 - modal_queries: INDISPONIBLE — organe modal absent : 0 classe logics.ml dans le jar 7a-java8-shade (inventaire zipfile) ; handler porte en siblings non vendus (normalizer, TweetyInitializer C186f) — note G.9 de argumenta

### Lecture du résultat : full

Le fill atteint 1,00 : les sept champs utiles de l'état sont peuplés, la
décision `fermeture_boutiques` est **dérivée** de la KB complète (les trois
formules, pas le seul fait initial — c'est toute la chaîne qui porte la
dérivation), et la conclusion 3 actes est écrite. Les deux dimensions
indisponibles ne comptent ni comme exécutées ni comme échecs : elles sont
rendues **avec leur preuve**. Vérifions-la nous-mêmes.

In [13]:
import zipfile

JAR = BASE.parent / "Tweety" / "libs" / "org.tweetyproject.tweety-7a-java8-shade.jar"
print("Jar inspecte :", JAR.name)
with zipfile.ZipFile(JAR) as jar:
    packages = sorted({
        n.rsplit("/", 1)[0]
        for n in jar.namelist()
        if n.endswith(".class") and "/logics/" in n
    })
print("Sous-packages logics du jar :")
for p in packages:
    print(" -", p.replace("/", "."))
fol = sum("logics/fol" in p for p in packages)
ml = sum("logics/ml" in p for p in packages)
print("Classes logics.fol :", fol, "| Classes logics.ml :", ml)

Jar inspecte : org.tweetyproject.tweety-7a-java8-shade.jar
Sous-packages logics du jar :
 - org.tweetyproject.logics.commons
 - org.tweetyproject.logics.commons.analysis
 - org.tweetyproject.logics.commons.analysis.streams
 - org.tweetyproject.logics.commons.error
 - org.tweetyproject.logics.commons.syntax
 - org.tweetyproject.logics.commons.syntax.interfaces
 - org.tweetyproject.logics.pl.analysis
 - org.tweetyproject.logics.pl.error
 - org.tweetyproject.logics.pl.parser
 - org.tweetyproject.logics.pl.postulates
 - org.tweetyproject.logics.pl.reasoner
 - org.tweetyproject.logics.pl.sat
 - org.tweetyproject.logics.pl.semantics
 - org.tweetyproject.logics.pl.syntax
 - org.tweetyproject.logics.pl.util
 - org.tweetyproject.logics.pl.writer
Classes logics.fol : 0 | Classes logics.ml : 0


### Lecture du résultat : la preuve d'indisponibilité

Le jar que charge la série ne porte que `logics.commons` et `logics.pl` :
aucune classe FOL, aucune classe modale. Les handlers portés dans
`argumentation_lib` le documentent eux-mêmes (notes G.9 du `__init__`) :
leurs dépendances amont — prover9, `TweetyInitializer`, le normalizer
modal — ne sont pas vendues dans cet arbre. Le sas EPITA exécutait FOL et
modale sur **ses** jars 1.28, dans un dossier étudiant que le volet B
s'interdit d'importer. La remontée attend les sous-tranches C186f/C186g :
jusque-là, le catalogue annonce, le run prouve, le carnet ne prétend pas.

### Exercice 3 — Construire la non-dérivation

La décision du banc est dérivée parce que la KB porte les trois formules.
Construisez un banc où la formule de décision **n'est plus dérivée** :
retirez la règle `detruit_proximite => fermeture_boutiques` de la KB et
relancez le niveau full. Que répond la dimension décision ? Quel champ du
fill-report change, le cas échéant ?

In [14]:
# Exercice a completer
# 1. banc_ex3 = dict(wl.BENCH) avec pl_belief_set sans la 2e regle.
# 2. Predire la reponse de la dimension decision.
decision_attendue = None  # TODO etudiant : True / False attendu
# 3. Relancer wl.run_level(banc_ex3, "full") et lire la note sat_decision.
print("Exercice a completer")

Exercice a completer


## Croisement avec le sas EPITA

L'organe `orchestration_modes.py` (carnet `07c`) porte les mesures du sas :
les fill-rates LLM par niveau et la parité de profondeur. Croisons-les avec
ce que le catalogue déterministe vient de produire.

In [15]:
ub_liste, cal_liste = om.RUNS["under_budget"], om.RUNS["calibrated"]

def fmt(v):
    # Un run non termine n'a pas de fill : le dire, jamais le fabriquer.
    return f"{v:.3f}" if v is not None else "  n/a"

print(f"{'mode':<24} {'phases':>6} {'fill UB':>8} {'fill cal':>9}")
for ub, cal in zip(ub_liste, cal_liste):
    print(f"{ub.mode:<24} {ub.phases_completed:>2}/{ub.phases_total:<3}"
          f" {fmt(ub.state_fill_rate):>8} {fmt(cal.state_fill_rate):>9}")
print("DEPTH_PARITY :", om.DEPTH_PARITY)

mode                     phases  fill UB  fill cal
pipeline_standard         6/15     0.196     0.462
pipeline_light            3/3      0.179     0.179
pipeline_full             7/17     0.196     0.487
conversational            1/3      0.096     0.308
conversation_deterministic  3/3        n/a       n/a
hierarchical_bridge       2/4      0.118       n/a
hierarchical_delegation   1/5        n/a       n/a
DEPTH_PARITY : (DepthParityRow(mode='pipeline_light', depth_dimension='workflow phases (DAG)', depth_count=3, nature='breadth', measured_range=None), DepthParityRow(mode='pipeline_standard', depth_dimension='workflow phases (DAG)', depth_count=15, nature='breadth', measured_range=None), DepthParityRow(mode='pipeline_full', depth_dimension='workflow phases (DAG)', depth_count=17, nature='breadth', measured_range=None), DepthParityRow(mode='hierarchical_bridge', depth_dimension='strategic objectives (default axes)', depth_count=4, nature='delegation', measured_range=None), DepthParityR

### Lecture du résultat : deux échelles, une asymétrie documentée

Les trois modes *pipeline* du sas montrent la gradation LLM : 0,179
(pipeline_light, sous budget), 0,462 (pipeline_standard calibré), 0,487
(pipeline_full calibré) — quand le catalogue déterministe produit
0,71 / 0,86 / 1,00. Les deux ne mesurent **pas** la même chose : le premier
compte des champs peuplés par des agents qui peuvent échouer, dévier ou
refuser (et certains runs du tableau rendent `n/a` — non terminé, jamais
fabriqué), le second des champs peuplés par des organes qui répondent
toujours. Les DAG de la source (3/15/17 phases) n'ont pas non plus de
contrepartie numérique ici (3/6/9 dimensions). L'asymétrie est documentée
dans `orchestration_modes.DEPTH_PARITY` et dans le contrat de fidélité de
`workflow_levels` : **portée, pas réparée** — maquiller une parité de
nombres serait exactement le contre-sens que le volet B refuse.

In [16]:
print(etat_full.final_conclusion)

Acte I — 4 arguments releves, KB posees : dung_extensions, pl. Acte II — 4 sophisme(s) releve(s), AF attaquee. Acte III — 4 requetes formelles executees, verdict par semantique Dung. (format compact du contrat 3 actes ; renderer integral : carnet 08d)


### Lecture du résultat : les trois actes

Acte I ce qui est posé (arguments relevés, KB posées) ; Acte II ce qui est
attaqué (sophismes, AF) ; Acte III ce qui tient (requêtes exécutées, verdict
par sémantique). C'est le **format compact** du contrat 3 actes — le
renderer intégral (narration, rendu, mise en forme complète) vit dans le
carnet `08d`, où il est embarqué et démontré ; l'extraire vers la lib est le
refactoring à demander à la série source (copie pédagogique déclarée,
organ-first).

## Ce que ce carnet a établi

- **Un catalogue exécutable** : 9 dimensions dont 7 réellement exécutées
  ici — relevé, PL, sophismes symboliques, Dung (3 sémantiques), décision,
  synthèse — chaque dimension citant son organe.
- **Des absences prouvées** : FOL et modale déclarées indisponibles avec
  preuve d'inventaire du jar — pas de silence, pas de faux résultat.
- **Une gradation mesurée** : fill 0,71 / 0,86 / 1,00, lisible champ par
  champ dans le fill-report, et croisée avec les mesures LLM du sas sans
  prétendre à une parité d'échelle qui n'existe pas.

## Pour aller plus loin

- `Argumentation-08d` — le renderer 3 actes intégral (narration *gated*,
  mise en forme complète) dont ce carnet n'utilise que le format compact.
- `orchestration_modes` (carnet `07c`) — les 7 modes du sas, leurs mesures
  de coût et la parité de profondeur.
- La remontée FOL/modale : sous-tranches C186f (`TweetyInitializer`) et
  C186g (shim JPype) — le jour où un jar complet rejoint la série, les deux
  dimensions déclarées indisponibles deviennent exécutables sans changer
  le catalogue.